# Healthcare KG Population Intelligence — V3.1
**Statistical audit and reproducibility gate** | NHANES 2017–2018 | No GitHub writes

This notebook audits V3 rather than assuming V3 is correct. It (1) verifies the untouched baseline, (2) reconstructs the V3 cohort, (3) compares independently implemented Taylor and delete-one-PSU jackknife variances, (4) runs an **optional independent R `survey` check**, (5) quantifies missingness and sensitivity, and (6) exports a validation report. No Kerendia eligibility or revenue claims are produced.

**Important:** Agreement of Taylor and jackknife variance is an algebraic property for a stratified PSU-total estimator in this particular design, *not independent external validation*. For external software validation use the R check below.


In [1]:
%pip -q install pandas numpy scipy requests networkx pyreadstat

## 1. Upload the V3 output ZIP (including `baseline_kg_COPY.json`)

In [2]:
from pathlib import Path
import hashlib, json, zipfile, datetime
import numpy as np, pandas as pd, requests
from scipy.stats import t as tdist
from google.colab import files
ROOT=Path('/content/Healthcare_KG_Population_Intelligence_V3_1'); ROOT.mkdir(exist_ok=True)
up=files.upload(); zp=Path(next(iter(up)))
with zipfile.ZipFile(zp) as archive:
    for name in archive.namelist():
        if Path(name).name in {'baseline_kg_COPY.json','v3_validation_report.json','v3_source_manifest.csv','v3_population_segments.csv'}:
            (ROOT/Path(name).name).write_bytes(archive.read(name))
kg=ROOT/'baseline_kg_COPY.json'; assert kg.exists(), 'Baseline missing'
kg_hash=hashlib.sha256(kg.read_bytes()).hexdigest()
prev=json.loads((ROOT/'v3_validation_report.json').read_text())
assert kg_hash==prev['baseline_sha256'], 'Baseline differs from V3'
print('Baseline SHA-256:',kg_hash)


Saving drive-download-20261025T020754Z-1-001.zip to drive-download-20261025T020754Z-1-001.zip
Baseline SHA-256: 4b43634ec370b1a7b1dfc28e6fbb9458172ba8234959d6c5f5713ad05d6212bf


## 2. Rebuild the identical V3 analysis cohort
The next cells are copied from V3 to preserve comparability. No newer NHANES cycles are mixed in.

In [3]:
BASE='https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/'
SOURCES={k:BASE+k+'.xpt' for k in ['DEMO_J','ALB_CR_J','BIOPRO_J','DIQ_J']}
tables={}; manifest=[]
for name,url in SOURCES.items():
    r=requests.get(url,timeout=90);r.raise_for_status()
    content=r.content; sha=hashlib.sha256(content).hexdigest()
    path=ROOT/(name+'.xpt');path.write_bytes(content)
    tables[name]=pd.read_sas(path,format='xport')
    manifest.append({'source':name,'url':url,'sha256':sha,'rows':len(tables[name])})
pd.DataFrame(manifest).to_csv(ROOT/'v3_1_source_manifest.csv',index=False)
print('Retrieved:',[(k,len(v)) for k,v in tables.items()])


Retrieved: [('DEMO_J', 9254), ('ALB_CR_J', 7936), ('BIOPRO_J', 6401), ('DIQ_J', 8897)]


In [6]:
d=tables['DEMO_J'][['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU']].copy()
for name in ['ALB_CR_J','BIOPRO_J','DIQ_J']:
    d=d.merge(tables[name],on='SEQN',how='left',validate='one_to_one')
d=d.loc[d.RIDAGEYR>=20].copy()
cols=['RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU','URXUMA','URXUCR','LBXSCR','DIQ010','DID040']
assert all(c in d for c in cols)
for c in cols:d[c]=pd.to_numeric(d[c],errors='coerce')
female=d.RIAGENDR.eq(2); kap=np.where(female,.7,.9); alpha=np.where(female,-.241,-.302)
r=d.LBXSCR/kap
d['egfr']=142*np.minimum(r,1)**alpha*np.maximum(r,1)**(-1.2)*(.9938**d.RIDAGEYR)*np.where(female,1.012,1)
d.loc[d.LBXSCR.isna()|d.LBXSCR.le(0),'egfr']=np.nan
d['uacr']=100*d.URXUMA/d.URXUCR
d.loc[d.URXUCR.isna()|d.URXUCR.le(0)|d.URXUMA.isna(),'uacr']=np.nan
d['ckd_screen']=np.where(d.egfr.notna()&d.uacr.notna(),(d.egfr<60)|(d.uacr>=30),np.nan)
d['diabetes_reported']=np.where(d.DIQ010.isin([1,2]),d.DIQ010.eq(1),np.nan)
d['t2d_proxy']=np.where(d.diabetes_reported==0,0,np.where((d.diabetes_reported==1)&d.DID040.between(1,100),(d.DID040>=30).astype(float),np.nan))
d['joint']=np.where(pd.notna(d.ckd_screen)&pd.notna(d.t2d_proxy),(d.ckd_screen==1)&(d.t2d_proxy==1),np.nan)
d['potassium']=pd.to_numeric(d.LBXSKSI,errors='coerce') if 'LBXSKSI' in d else np.nan
d['age_group']=pd.cut(d.RIDAGEYR,[20,45,65,75,np.inf],right=False,labels=['20-44','45-64','65-74','75+'])
d['sex']=d.RIAGENDR.map({1:'Male',2:'Female'})
d['g_cat']=pd.cut(d.egfr,[-np.inf,15,30,45,60,90,np.inf],right=False,labels=['G5','G4','G3b','G3a','G2','G1'])
d['a_cat']=pd.cut(d.uacr,[0,30,300,np.inf],right=False,labels=['A1','A2','A3'])
design=d.WTMEC2YR.gt(0)&d.SDMVSTRA.notna()&d.SDMVPSU.notna()
complete=d.joint.notna()&design
print('Adults:',len(d),'complete:',int(complete.sum()),'positive:',int((d.joint==1).sum()))
assert int(complete.sum())==prev['denominator_reconciliation']['complete_case_unweighted']
assert int((d.joint==1).sum())==prev['denominator_reconciliation']['positive_unweighted']

Adults: 5569 complete: 4662 positive: 314


In [7]:
# Print columns present in each table to locate the missing columns
for name, df_table in tables.items():
    print(f"{name} columns:", [col for col in ['LBXSCR', 'URXUMA', 'URXUCR', 'DIQ010', 'DIQ040'] if col in df_table.columns])
print("Columns currently in d:", d.columns.tolist())

DEMO_J columns: []
ALB_CR_J columns: ['URXUMA', 'URXUCR']
BIOPRO_J columns: ['LBXSCR']
DIQ_J columns: ['DIQ010']
Columns currently in d: ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU', 'URXUMA', 'URXUMS', 'URDUMALC', 'URXUCR', 'URXCRS', 'URDUCRLC', 'URDACT', 'LBXSATSI', 'LBDSATLC', 'LBXSAL', 'LBDSALSI', 'LBXSAPSI', 'LBXSASSI', 'LBXSC3SI', 'LBXSBU', 'LBDSBUSI', 'LBXSCLSI', 'LBXSCK', 'LBXSCR', 'LBDSCRSI', 'LBXSGB', 'LBDSGBSI', 'LBXSGL', 'LBDSGLSI', 'LBXSGTSI', 'LBDSGTLC', 'LBXSIR', 'LBDSIRSI', 'LBXSLDSI', 'LBXSOSSI', 'LBXSPH', 'LBDSPHSI', 'LBXSKSI', 'LBXSNASI', 'LBXSTB', 'LBDSTBSI', 'LBDSTBLC', 'LBXSCA', 'LBDSCASI', 'LBXSCH', 'LBDSCHSI', 'LBXSTP', 'LBDSTPSI', 'LBXSTR', 'LBDSTRSI', 'LBXSUA', 'LBDSUASI', 'DIQ010', 'DID040', 'DIQ160', 'DIQ170', 'DIQ172', 'DIQ175A', 'DIQ175B', 'DIQ175C', 'DIQ175D', 'DIQ175E', 'DIQ175F', 'DIQ175G', 'DIQ175H', 'DIQ175I', 'DIQ175J', 'DIQ175K', 'DIQ175L', 'DIQ175M', 'DIQ175N', 'DIQ175O', 'DIQ175P', 'DIQ175Q', 'DIQ175R', 'DIQ175S',

## 3. Statistical audit: Taylor vs delete-one-PSU jackknife
Compute each estimator from PSU-level contributions using separate functions. The total estimator includes all adult design PSUs, assigning zero to noncases. For prevalence, use the linearized ratio residual `case − prevalence × complete_case`. Confidence intervals use design degrees of freedom.

**Interpretation:** Exact or near-exact variance agreement is expected for this particular *linearized* total; it does not prove that either method is externally correct. The optional R `survey` test below is the independent check.

In [8]:
z=d.loc[design].copy()
z['weight']=z.WTMEC2YR
strata_psus=z.groupby('SDMVSTRA',observed=True)['SDMVPSU'].nunique()
dfree=int((strata_psus-1).clip(lower=0).sum())
assert dfree>0

def taylor_total_var(y):
    x=z[['SDMVSTRA','SDMVPSU']].copy()
    x['weighted']=z.weight*y.reindex(z.index).fillna(0).astype(float)
    grouped=x.groupby(['SDMVSTRA','SDMVPSU'],observed=True).weighted.sum()
    result=0.0
    for _,a in grouped.groupby(level=0):
        vals=a.to_numpy();m=len(vals)
        if m>1: result+=m/(m-1)*np.sum((vals-vals.mean())**2)
    return float(result)

def jackknife_total_var(y):
    x=z[['SDMVSTRA','SDMVPSU']].copy()
    x['weighted']=z.weight*y.reindex(z.index).fillna(0).astype(float)
    grouped=x.groupby(['SDMVSTRA','SDMVPSU'],observed=True).weighted.sum()
    result=0.0
    for _,a in grouped.groupby(level=0):
        vals=a.to_numpy();m=len(vals)
        if m>1:
            replicate_totals=(vals.sum()-vals)*m/(m-1)
            result+=(m-1)/m*np.sum((replicate_totals-replicate_totals.mean())**2)
    return float(result)

case=(complete & d.joint.eq(1)).astype(float)
denom=complete.astype(float)
N=float((z.weight*case.reindex(z.index)).sum())
D=float((z.weight*denom.reindex(z.index)).sum())
p=N/D
residual=case-p*denom
v_t=taylor_total_var(case);v_j=jackknife_total_var(case)
v_ratio=taylor_total_var(residual)
crit=tdist.ppf(.975,dfree)
results={'weighted_cases':N,'weighted_complete_case_denominator':D,'prevalence':p,
         'taylor_variance_total':v_t,'jackknife_variance_total':v_j,
         'variance_relative_difference':abs(v_t-v_j)/v_t,
         'count_ci_95':[max(0,N-crit*np.sqrt(v_t)),N+crit*np.sqrt(v_t)],
         'prevalence_ci_95':[max(0,p-crit*np.sqrt(v_ratio)/D),min(1,p+crit*np.sqrt(v_ratio)/D)],
         'design_df':dfree}
assert np.isclose(v_t,v_j,rtol=1e-8)
print(json.dumps(results,indent=2))


{
  "weighted_cases": 9497619.755203877,
  "weighted_complete_case_denominator": 216820702.8793857,
  "prevalence": 0.04380402622570257,
  "taylor_variance_total": 905053720442.1208,
  "jackknife_variance_total": 905053720442.1208,
  "variance_relative_difference": 0.0,
  "count_ci_95": [
    7469879.907324279,
    11525359.603083475
  ],
  "prevalence_ci_95": [
    0.0360324311128765,
    0.05157562133852864
  ],
  "design_df": 15
}


## 4. Optional independent cross-check with R `survey`
This cell installs R and the established `survey` package in Colab (may take several minutes). It checks the case **total** and complete-case **ratio**. A failure to install R is recorded as **NOT RUN**, not PASS. NHANES design variables use masked strata and PSU identifiers with examination weights.

In [10]:
import subprocess, shutil
rdata=z[['SDMVSTRA','SDMVPSU','WTMEC2YR']].copy()
rdata['case']=case.reindex(z.index).astype(int)
rdata['complete']=denom.reindex(z.index).astype(int)
rdata.to_csv(ROOT/'v3_1_r_crosscheck_input.csv',index=False)
r_script=r"""suppressPackageStartupMessages(library(survey))
d <- read.csv(commandArgs(trailingOnly=TRUE)[1])
design <- svydesign(ids=~SDMVPSU,strata=~SDMVSTRA,weights=~WTMEC2YR,nest=TRUE,data=d)
ct <- svytotal(~case,design)
rat <- svyratio(~case,~complete,design)
cat(sprintf("r_count,%.12f\n",coef(ct)[1]))
cat(sprintf("r_count_variance,%.12f\n",vcov(ct)[1,1]))
cat(sprintf("r_prevalence,%.12f\n",coef(rat)[1]))
cat(sprintf("r_prevalence_variance,%.12f\n",vcov(rat)[1,1]))
"""
(ROOT/'v3_1_r_crosscheck.R').write_text(r_script)
r_check={'status':'NOT RUN','reason':'R survey package not available'}
try:
    if not shutil.which('Rscript'):
        subprocess.run(['apt-get','update','-qq'],check=True,stdout=subprocess.DEVNULL,timeout=180)
        subprocess.run(['apt-get','install','-y','-qq','r-base-core','r-cran-survey'],check=True,stdout=subprocess.DEVNULL,timeout=240)
    elif subprocess.run(['Rscript','-e','library(survey)'],capture_output=True).returncode:
        subprocess.run(['apt-get','update','-qq'],check=True,stdout=subprocess.DEVNULL,timeout=180)
        subprocess.run(['apt-get','install','-y','-qq','r-cran-survey'],check=True,stdout=subprocess.DEVNULL,timeout=240)
    run=subprocess.run(['Rscript',str(ROOT/'v3_1_r_crosscheck.R'),str(ROOT/'v3_1_r_crosscheck_input.csv')],capture_output=True,text=True,check=True,timeout=90)
    rvals={k:float(v) for k,v in (line.strip().split(',') for line in run.stdout.splitlines() if line.startswith('r_'))}
    matches={'count':bool(np.isclose(rvals['r_count'],N,rtol=1e-7)),
             'count_variance':bool(np.isclose(rvals['r_count_variance'],v_t,rtol=1e-6)),
             'prevalence':bool(np.isclose(rvals['r_prevalence'],p,rtol=1e-7)),
             'prevalence_variance':bool(np.isclose(rvals['r_prevalence_variance'],v_ratio/D**2,rtol=1e-6))}
    r_check={'status':'PASS' if all(matches.values()) else 'FAIL','values':rvals,'matches':matches}
except Exception as exc:
    r_check={'status':'NOT RUN','reason':str(exc)[:600]}
print(json.dumps(r_check,indent=2))

{
  "status": "PASS",
  "values": {
    "r_count": 9497619.755203877,
    "r_count_variance": 905053720442.121,
    "r_prevalence": 0.043804026226,
    "r_prevalence_variance": 1.3294445e-05
  },
  "matches": {
    "count": true,
    "count_variance": true,
    "prevalence": true,
    "prevalence_variance": true
  }
}


## 5. Missingness by age, sex and race/ethnicity
Use the **full adult MEC-weighted domain** as the denominator in each group. The joint-screening proxy is missing if either component cannot be evaluated. Weighted missingness is descriptive, not an imputation model.

In [11]:
d['observed_joint']=complete
missing_rows=[]
for field in ['age_group','sex','RIDRETH3']:
    for value in d[field].dropna().unique():
        subset=design & d[field].eq(value)
        denom_w=float(d.loc[subset,'WTMEC2YR'].sum())
        missing_mask=subset & ~complete
        missing_rows.append({'dimension':field,'segment':str(value),
           'adult_n':int(subset.sum()),'missing_n':int(missing_mask.sum()),
           'adult_weighted':denom_w,'missing_weighted':float(d.loc[missing_mask,'WTMEC2YR'].sum()),
           'weighted_missing_fraction':float(d.loc[missing_mask,'WTMEC2YR'].sum())/denom_w if denom_w else np.nan})
miss=pd.DataFrame(missing_rows)
miss.to_csv(ROOT/'v3_1_missingness_by_group.csv',index=False)
print(miss.to_string(index=False))


dimension segment  adult_n  missing_n  adult_weighted  missing_weighted  weighted_missing_fraction
age_group   65-74      819        147    2.877327e+07      2.840908e+06                   0.098734
age_group     75+      681        165    1.984561e+07      2.993113e+06                   0.150820
age_group   45-64     1974        289    8.505768e+07      7.352140e+06                   0.086437
age_group   20-44     2095        306    1.050610e+08      8.730675e+06                   0.083101
      sex  Female     2867        444    1.238266e+08      1.134596e+07                   0.091628
      sex    Male     2702        463    1.149110e+08      1.057088e+07                   0.091992
 RIDRETH3     4.0     1298        262    2.733251e+07      4.379571e+06                   0.160233
 RIDRETH3     6.0      811        172    1.406591e+07      2.097640e+06                   0.149129
 RIDRETH3     3.0     1935        296    1.486010e+08      1.225284e+07                   0.082455
 RIDRETH3 

## 6. Missingness sensitivity — explicitly bounded, not corrected
Report extreme lower and upper bounds by classifying all missing joint outcomes as negative or positive. These are deliberately unrealistic stress tests. Complete-case prevalence and all-adult bounds use different denominators; they must not be presented interchangeably.

In [12]:
adult_weight=float(d.loc[design,'WTMEC2YR'].sum())
missing_weight=float(d.loc[design & ~complete,'WTMEC2YR'].sum())
sensitivity=pd.DataFrame([
 {'scenario':'Complete-case conditional prevalence','numerator':N,'denominator':D,'prevalence':N/D},
 {'scenario':'All missing are noncases (lower extreme)','numerator':N,'denominator':adult_weight,'prevalence':N/adult_weight},
 {'scenario':'All missing are cases (upper extreme)','numerator':N+missing_weight,'denominator':adult_weight,'prevalence':(N+missing_weight)/adult_weight},
])
sensitivity.to_csv(ROOT/'v3_1_missingness_bounds.csv',index=False)
print(sensitivity.to_string(index=False))


                                scenario    numerator  denominator  prevalence
    Complete-case conditional prevalence 9.497620e+06 2.168207e+08    0.043804
All missing are noncases (lower extreme) 9.497620e+06 2.387375e+08    0.039783
   All missing are cases (upper extreme) 3.141446e+07 2.387375e+08    0.131586


## 7. Audit report and gates
`PASS` means internal calculations and (if available) R agree. It does **not** establish a nationally validated CKD/T2D prevalence estimate. Missingness, single-measure CKD, imperfect T2D proxy, noninstitutionalized population coverage, and potential laboratory calibration remain substantive limitations.

In [13]:
audit={'version':'3.1','timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'baseline_sha256':kg_hash,'baseline_unchanged':True,'github_accessed':False,
 'cohort':{'adults':len(d),'complete_cases':int(complete.sum()),'joint_cases':int(case.sum())},
 'estimates':results,'r_survey_crosscheck':r_check,
 'missingness':{'adult_weight':adult_weight,'complete_weight':D,'missing_weight':missing_weight,
                'missing_fraction_weighted':missing_weight/adult_weight},
 'statistical_gate':'PASS' if r_check['status']=='PASS' else 'INCOMPLETE',
 'limitations':['2017–2018 only','one-time kidney screening does not establish chronic CKD',
  'age-at-diagnosis proxy cannot confirm type 2 diabetes','complete-case estimates may be biased',
  'extreme missingness bounds are not corrections','no individual Kerendia eligibility determination',
  'small subgroups may be unstable','not representative of institutionalized populations']}
(ROOT/'v3_1_audit_report.json').write_text(json.dumps(audit,indent=2))
assert hashlib.sha256(kg.read_bytes()).hexdigest()==kg_hash
print('Statistical gate:',audit['statistical_gate'])
print('Outputs:',*[p.name for p in ROOT.glob('v3_1_*')],sep='\n  ')


Statistical gate: PASS
Outputs:
  v3_1_audit_report.json
  v3_1_missingness_by_group.csv
  v3_1_r_crosscheck.R
  v3_1_r_crosscheck_input.csv
  v3_1_missingness_bounds.csv
  v3_1_source_manifest.csv


## 8. Export to Google Drive (optional)
Only development files are written to a new V3.1 directory. No GitHub credentials or repository operations are used.

In [14]:
from google.colab import drive
drive.mount('/content/drive')
target=Path('/content/drive/MyDrive/Healthcare_KG_Population_Intelligence_V3_1');target.mkdir(parents=True,exist_ok=True)
import shutil
for f in ROOT.glob('v3_1_*'):
    if f.suffix in {'.json','.csv','.R'}:shutil.copy2(f,target/f.name)
shutil.copy2(kg,target/kg.name)
print('Saved to',target)


Mounted at /content/drive
Saved to /content/drive/MyDrive/Healthcare_KG_Population_Intelligence_V3_1


### V4 readiness decision
Proceed to newer NHANES cycles **only after** inspecting the V3.1 audit report, resolving any independent R validation failure, and documenting how missingness affects interpretation. V3.1 intentionally does not change the baseline graph or production workflow.